# Week 10 Model Answer: Robust RSAM Computation from Problematic SDS Archives

This notebook is a model answer for the Week 10 homework exercise. The goal is not to prescribe a single perfect preprocessing workflow, but to demonstrate a robust, defensible strategy for computing RSAM from messy continuous seismic datasets.

The notebook includes:

- a switch between the datasets used by students,
- one-day-at-a-time SDS reading,
- buffered reading to reduce edge effects,
- optional vertical-component selection,
- robust percentile clipping,
- baseline stabilization,
- detrending, tapering, and filtering,
- RSAM computation and output,
- plotting and output compression.

The central lesson is that real observatory data are often ill-conditioned. Dropouts, spikes, gaps, and baseline changes can strongly affect RSAM, so preprocessing choices must be made carefully and documented clearly.


## 1. Imports

This assumes the course environment has `flovopy`, `obspy`, and the Week 8 helper `set_samba_data_root.py` available.

If running on a different machine, update the data-root setup cell below.


In [ ]:
from pathlib import Path
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt

from obspy import UTCDateTime, Stream

# Course/FLOVOpy imports.
# Adjust these imports if the package layout changes.
from flovopy.enhanced.sdsclient import EnhancedSDSClient
from flovopy.processing.sam import RSAM


## 2. Data-root setup

In class, the SDS archives were accessed from the mounted class data directory. The exact path may differ between local machines and `newton`.

The preferred approach is to process data directly from the mounted archive, rather than copying large SDS archives locally.


In [ ]:
# Option A: use the helper from the course notebooks.
try:
    import sys
    sys.path.append("../week8")
    from set_samba_data_root import DATA_ROOT
    DATA_ROOT = Path(DATA_ROOT)
except Exception:
    # Option B: edit this manually if needed.
    DATA_ROOT = Path("/Volumes/classdata")

print(f"DATA_ROOT = {DATA_ROOT}")
print(f"Exists? {DATA_ROOT.exists()}")


## 3. Dataset switch

Choose one of the datasets below by changing `DATASET_NAME`.

The date ranges are examples and should be adjusted depending on data availability and the specific homework target.


In [ ]:
DATASETS = {
    "sakurajima": {
        "description": "Sakurajima SDS archive",
        "sds_root": DATA_ROOT / "SDS_Sakurajima",
        "start": UTCDateTime(2015, 6, 1),
        "end": UTCDateTime(2015, 6, 14),
        "network": "*",
        "station": "*",
        "location": "*",
        "channel": "*",
    },

    "nevado_gcf": {
        "description": "Nevado del Ruiz GCF-derived SDS archive",
        "sds_root": DATA_ROOT / "NevadoDelRuiz" / "SDS_GCF",
        "start": UTCDateTime(2012, 4, 2),
        "end": UTCDateTime(2012, 4, 16),
        "network": "*",
        "station": "*",
        "location": "*",
        "channel": "*",
    },

    "ksc_2026": {
        "description": "KSC 2026 service dataset",
        "sds_root": DATA_ROOT / "KSC_2026" / "20260310_service",
        "start": UTCDateTime(2026, 1, 8),
        "end": UTCDateTime(2026, 1, 22),
        "network": "*",
        "station": "*",
        "location": "*",
        "channel": "*",
    },
}

# Choose one: "sakurajima", "nevado_gcf", or "ksc_2026"
DATASET_NAME = "sakurajima"

cfg = DATASETS[DATASET_NAME]
SDS_ROOT = cfg["sds_root"]
START = cfg["start"]
END = cfg["end"]

OUTPUT_ROOT = Path.home() / "work" / "CompSciS26" / "week10_model_answer"
SAM_DIR = OUTPUT_ROOT / DATASET_NAME / "RSAM"
SAM_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Description: {cfg['description']}")
print(f"SDS_ROOT: {SDS_ROOT}")
print(f"SDS_ROOT exists? {SDS_ROOT.exists()}")
print(f"START: {START}")
print(f"END:   {END}")
print(f"SAM_DIR: {SAM_DIR}")


## 4. Inspect archive availability

Before processing, check which SEED IDs are available. This helps decide whether we should process all channels or restrict to vertical components.


In [ ]:
client = EnhancedSDSClient(SDS_ROOT)

try:
    availdf, seed_ids = client.get_availability(
        startday=START,
        endday=END,
        verbose=True,
    )
    display(availdf)
    print(f"Number of SEED IDs: {len(seed_ids)}")
    print(seed_ids[:20])
except Exception as e:
    print(f"Availability check failed: {e}")
    availdf = None
    seed_ids = []


## 5. Defensive preprocessing utilities

These functions are intentionally conservative. They are designed to make RSAM computation more robust for data with spikes, gaps, dropouts, and baseline shifts.

Important caveat: aggressive cleaning can remove real signal. The goal here is to stabilize a first-pass RSAM product, not to produce a final publishable waveform-processing method.


In [ ]:
def trace_is_vertical(tr):
    """Return True if a trace appears to be a vertical component."""
    return tr.stats.channel.upper().endswith("Z")


def clip_trace_percentile(tr, pct=99.9, multiplier=2.0):
    """
    Clip large-amplitude spikes using a robust percentile threshold.

    This is useful when a few extreme samples would otherwise dominate RSAM.
    """
    tr = tr.copy()
    data = tr.data.astype(float)

    if data.size == 0:
        return tr

    threshold = np.nanpercentile(np.abs(data), pct) * multiplier

    if not np.isfinite(threshold) or threshold == 0:
        return tr

    tr.data = np.clip(data, -threshold, threshold).astype(float)
    return tr


def remove_trace_baseline_approximately(tr):
    """
    Apply a conservative baseline stabilization step.

    Median removal is often safer than mean removal when spikes are present.
    This will not perfectly remove step-like baseline shifts, but it is a useful
    first-pass correction for problematic data.
    """
    tr = tr.copy()
    data = tr.data.astype(float)

    if data.size == 0:
        return tr

    tr.data = data - np.nanmedian(data)
    return tr


def preprocess_stream(
    st,
    use_vertical_only=False,
    clip=True,
    clip_pct=99.9,
    clip_multiplier=2.0,
    detrend=True,
    taper=True,
    filter_data=True,
    freqmin=0.5,
    freqmax=None,
):
    """
    Defensive preprocessing for ill-conditioned continuous seismic data.

    Order used here:
      1. optional vertical-channel selection
      2. merge with gap handling
      3. median baseline removal
      4. percentile clipping
      5. detrend
      6. taper
      7. filter

    This is not the only valid order. It is a reasonable robust strategy for a
    first-pass RSAM product from messy field data.
    """
    st = st.copy()

    if use_vertical_only:
        st = Stream([tr for tr in st if trace_is_vertical(tr)])

    if len(st) == 0:
        return st

    try:
        st.merge(method=1, fill_value="latest")
    except Exception as e:
        print(f"Merge warning: {e}")

    processed = Stream()

    for tr in st:
        if tr.stats.npts == 0:
            continue

        tr = remove_trace_baseline_approximately(tr)

        if clip:
            tr = clip_trace_percentile(tr, pct=clip_pct, multiplier=clip_multiplier)

        if detrend:
            try:
                tr.detrend("linear")
                tr.detrend("demean")
            except Exception as e:
                print(f"Detrend warning for {tr.id}: {e}")

        if taper:
            try:
                tr.taper(max_percentage=0.01, type="hann")
            except Exception as e:
                print(f"Taper warning for {tr.id}: {e}")

        if filter_data:
            try:
                if freqmax is None:
                    tr.filter("highpass", freq=freqmin, corners=2, zerophase=True)
                else:
                    tr.filter("bandpass", freqmin=freqmin, freqmax=freqmax, corners=2, zerophase=True)
            except Exception as e:
                print(f"Filter warning for {tr.id}: {e}")

        processed.append(tr)

    return processed


## 6. Quick single-day test

Before processing many days, test one day. This is especially important for messy datasets.


In [ ]:
TEST_DAY = START
READ_BUFFER_SECONDS = 3600

read_start = TEST_DAY - READ_BUFFER_SECONDS
read_end = TEST_DAY + 86400 + READ_BUFFER_SECONDS

print(f"Reading test window: {read_start} to {read_end}")

st_test = client.get_waveforms(
    network=cfg["network"],
    station=cfg["station"],
    location=cfg["location"],
    channel=cfg["channel"],
    starttime=read_start,
    endtime=read_end,
)

print(st_test)
print(f"Unique SEED IDs: {len(set(tr.id for tr in st_test))}")


In [ ]:
use_vertical_only = len(set(tr.id for tr in st_test)) > 10

st_clean = preprocess_stream(
    st_test,
    use_vertical_only=use_vertical_only,
    clip=True,
    clip_pct=99.9,
    clip_multiplier=2.0,
    detrend=True,
    taper=True,
    filter_data=True,
    freqmin=0.5,
    freqmax=None,
)

st_clean.trim(TEST_DAY, TEST_DAY + 86400)
st_clean = Stream([tr for tr in st_clean if tr.stats.npts > 0])

print(st_clean)


## 7. Plot before/after examples

This helps diagnose whether preprocessing is helping or making things worse.


In [ ]:
def plot_first_trace_before_after(st_raw, st_processed):
    if len(st_raw) == 0 or len(st_processed) == 0:
        print("No traces to plot.")
        return

    raw = st_raw[0].copy()
    proc = st_processed.select(id=raw.id)

    if len(proc) == 0:
        proc = st_processed[0]
    else:
        proc = proc[0]

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)

    axes[0].plot(raw.times("matplotlib"), raw.data, linewidth=0.5)
    axes[0].set_title(f"Raw trace: {raw.id}")
    axes[0].set_ylabel("Counts")

    axes[1].plot(proc.times("matplotlib"), proc.data, linewidth=0.5)
    axes[1].set_title(f"Processed trace: {proc.id}")
    axes[1].set_ylabel("Processed counts")

    plt.tight_layout()
    plt.show()


plot_first_trace_before_after(st_test, st_clean)


## 8. RSAM computation function

This reads one day at a time using a buffered window, preprocesses the data, trims back to the target day, computes RSAM, and writes output files.


In [ ]:
def compute_rsam_for_dataset(
    client,
    start,
    end,
    sam_dir,
    network="*",
    station="*",
    location="*",
    channel="*",
    sampling_interval=60.0,
    read_buffer_seconds=3600,
    clip_pct=99.9,
):
    """
    Compute RSAM one UTC day at a time.

    The extra read buffer reduces filtering/tapering edge effects. Data are
    trimmed back to the target day after preprocessing.
    """
    day = UTCDateTime(start.year, start.month, start.day)

    while day < end:
        day_start = day
        day_end = min(day + 86400, end)

        read_start = day_start - read_buffer_seconds
        read_end = day_end + read_buffer_seconds

        print("=" * 80)
        print(f"Processing target day: {day_start.date}")
        print(f"Read window: {read_start} to {read_end}")

        try:
            st = client.get_waveforms(
                network=network,
                station=station,
                location=location,
                channel=channel,
                starttime=read_start,
                endtime=read_end,
            )
        except Exception as e:
            print(f"Read failed for {day_start.date}: {e}")
            day += 86400
            continue

        if len(st) == 0:
            print("No data found.")
            day += 86400
            continue

        seed_ids = sorted(set(tr.id for tr in st))
        use_vertical_only = len(seed_ids) > 10

        print(f"Read {len(st)} traces; {len(seed_ids)} unique SEED IDs")
        if use_vertical_only:
            print("More than 10 SEED IDs found: selecting vertical components only.")

        st = preprocess_stream(
            st,
            use_vertical_only=use_vertical_only,
            clip=True,
            clip_pct=clip_pct,
            clip_multiplier=2.0,
            detrend=True,
            taper=True,
            filter_data=True,
            freqmin=0.5,
            freqmax=None,
        )

        if len(st) == 0:
            print("No traces remain after preprocessing.")
            day += 86400
            continue

        st.trim(day_start, day_end)
        st = Stream([tr for tr in st if tr.stats.npts > 0])

        if len(st) == 0:
            print("No data remain after final trim.")
            day += 86400
            continue

        try:
            rsam = RSAM(stream=st, sampling_interval=sampling_interval)
            rsam.write(SAM_DIR=str(sam_dir), ext="csv")
            print(f"Wrote RSAM for {day_start.date}")
        except Exception as e:
            print(f"RSAM failed for {day_start.date}: {e}")

        day += 86400


## 9. Run RSAM computation

This may take a while depending on the dataset, network mount, and number of channels.

For a quick test, reduce `END` above to only a few days.


In [ ]:
compute_rsam_for_dataset(
    client=client,
    start=START,
    end=END,
    sam_dir=SAM_DIR,
    network=cfg["network"],
    station=cfg["station"],
    location=cfg["location"],
    channel=cfg["channel"],
    sampling_interval=60.0,
    read_buffer_seconds=3600,
    clip_pct=99.9,
)


## 10. Read RSAM back and plot

Reading the RSAM back from disk verifies that the output files were written in a reusable form.


In [ ]:
try:
    rsam = RSAM.read(
        START,
        END,
        SAM_DIR=str(SAM_DIR),
        ext="csv",
    )

    print(rsam)
    rsam.plot(metrics="median")
except Exception as e:
    print(f"Failed to read or plot RSAM: {e}")


## 11. Optional: compare different RSAM metrics

Depending on the implementation of the `RSAM` class, other metrics may be available, such as mean, median, maximum, minimum, or frequency-ratio-style products.


In [ ]:
# Example only. Uncomment and adjust depending on available RSAM metrics.
# rsam.plot(metrics=["mean", "median", "max"])


## 12. Compress RSAM outputs

This creates a ZIP archive of the RSAM output directory for submission or transfer.


In [ ]:
zip_base = SAM_DIR.parent / f"{DATASET_NAME}_rsam_week10_model"
zip_file = shutil.make_archive(str(zip_base), "zip", SAM_DIR)

print(f"Saved compressed RSAM archive to:
{zip_file}")


## 13. Reflection: why this workflow is defensible

A good Week 10 answer should explain the processing choices. For example:

1. I processed one day at a time to avoid loading too much data at once.
2. I read an extra buffer before and after each day so that filtering and tapering artifacts could be trimmed away.
3. I selected only vertical components when the archive contained many channels.
4. I used percentile clipping because a few pathological samples can dominate RSAM.
5. I merged traces carefully and avoided filling gaps with zeros.
6. I removed approximate baselines, detrended, tapered, and filtered before computing RSAM.
7. I saved the RSAM products to disk and read them back to verify the output.

This workflow is not perfect. In particular, damaged instruments with baseline jumps or complicated dropouts may require more advanced treatment. But this is a reasonable first-pass operational strategy and a good starting point for further development.


## 14. Possible improvements

Future versions could add:

- explicit gap masks,
- step detection and correction,
- MAD-based despiking,
- comparison of clipping thresholds,
- plots of raw vs cleaned RSAM,
- quality-control flags,
- per-station summaries,
- server-side batch processing on `newton`,
- parallel processing by day or station.
